Scritp to merge the parkinsons HOA - PD or alzhaimer HOA - AD datasets, to perform sleep staging benchmarks

In [ ]:
import os
import torch
import numpy as np
from tqdm import tqdm
# change working directory 
os.chdir( os.path.join( os.environ["VSC_DATA"], "physioex" ) )

from physioex.train.utils.fast_train import FastTrainDataset, FastEvalDataset

data_folder = f"{os.environ['VSC_SCRATCH_PROJECTS_BASE']}/2024_111/guido/"

DATASET = "parkinsons"
datasets = ["parkinsons/night/HOA", "parkinsons/night/PD"]

train_HOA = FastTrainDataset(
    datasets = [datasets[0]],
    preprocess = "xsleepnet",
    L = 21,
    indexed_channels = [0, 1, 2],
    data_folder = data_folder,
)

train_AD = FastTrainDataset(
    datasets = [datasets[1]],
    preprocess = "xsleepnet",
    L = 21,
    indexed_channels = [0, 1, 2],
    data_folder = data_folder,
)

# we need to get the scaling mean and std from the training set
scalingHOA = np.load( f"{data_folder}/{datasets[0]}/xsleepnet/scaling.npz" ) 
scalingAD = np.load( f"{data_folder}/{datasets[1]}/xsleepnet/scaling.npz" )

meanHOA, stdHOA = scalingHOA["mean"], scalingHOA["std"]
meanAD, stdAD = scalingAD["mean"], scalingAD["std"]

# convert to torch tensors
meanHOA = torch.tensor(meanHOA, dtype=torch.float32)
meanAD = torch.tensor(meanAD, dtype=torch.float32)
stdHOA = torch.tensor(stdHOA, dtype=torch.float32)
stdAD = torch.tensor(stdAD, dtype=torch.float32)

# now we want to invert-scale the data   
train_HOA.X = (train_HOA.X * stdHOA) + meanHOA
train_AD.X = (train_AD.X * stdAD) + meanAD

# now we merge the two datasets and create a new dataset

X = torch.cat( [train_HOA.X, train_AD.X], dim=0 )
y = torch.cat( [train_HOA.y, train_AD.y], dim=0 )

# we need now to scale the input data
mean, std = X.mean(dim=0), X.std(dim=0)
X = (X - mean) / std

# save the new dataset
torch.save( (X, y), f"{data_folder}/.tmp/{DATASET}/xsleepnet/train_dataset.pt" )

X, y = [], []
# we do the same for the evaluation dataset
eval_HOA = FastEvalDataset(
    datasets = [datasets[0]],
    preprocess = "xsleepnet",
    indexed_channels = [0, 1, 2],
    data_folder= data_folder,
    split= "eval",
)

for subject_data, subject_labels in zip(
    eval_HOA.X, eval_HOA.y
):
    # invert scale the data
    subject_data = (subject_data * stdHOA) + meanHOA
    # scale the data with the new mean and std
    subject_data = (subject_data - mean) / std

    X.append(subject_data)  
    y.append(subject_labels)

eval_AD = FastEvalDataset(
    datasets = [datasets[1]],
    preprocess = "xsleepnet",
    indexed_channels = [0, 1, 2],
    data_folder= data_folder,
    split= "eval",
)

for subject_data, subject_labels in zip(
    eval_AD.X, eval_AD.y
):
    # unscale the data
    subject_data = (subject_data * stdAD) + meanAD
    # scale the data with the new mean and std
    subject_data = (subject_data - mean) / std

    X.append(subject_data)  
    y.append(subject_labels)

# save the evaluation dataset
torch.save( (X, y), f"{data_folder}/.tmp/{DATASET}/xsleepnet/eval_dataset.pt" )

# do the same for the test dataset

test_HOA = FastEvalDataset(
    datasets = [datasets[0]],
    preprocess = "xsleepnet",
    indexed_channels = [0, 1, 2],
    data_folder= data_folder,
    split= "test",
)
X, y = [], []
for subject_data, subject_labels in zip(
    test_HOA.X, test_HOA.y
):
    # unscale the data
    subject_data = (subject_data * stdHOA) + meanHOA
    # scale the data with the new mean and std
    subject_data = (subject_data - mean) / std

    X.append(subject_data)  
    y.append(subject_labels)

test_AD = FastEvalDataset(
    datasets = [datasets[1]],
    preprocess = "xsleepnet",
    indexed_channels = [0, 1, 2],
    data_folder= data_folder,
    split= "test",
)

for subject_data, subject_labels in zip(
    test_AD.X, test_AD.y
):
    # unscale the data
    subject_data = (subject_data * stdAD) + meanAD
    # scale the data with the new mean and std
    subject_data = (subject_data - mean) / std

    X.append(subject_data)  
    y.append(subject_labels)

# save the test dataset
torch.save( (X, y), f"{data_folder}/.tmp/{DATASET}/xsleepnet/test_dataset.pt" )

# save also the mean and std used for scaling
torch.save(
    (mean, std),
    f"{data_folder}/.tmp/{DATASET}/xsleepnet/scaling.pt"
)



2025-06-12 13:12:14.099 | INFO     | physioex.train.utils.fast_train:__init__:32 - Loading FastTrainDataset for parkinsons/night/HOA with preprocess xsleepnet
2025-06-12 13:12:15.562 | INFO     | physioex.train.utils.fast_train:__init__:45 - Loaded X: torch.Size([30240, 3, 29, 129]), y: torch.Size([30240])
2025-06-12 13:12:15.635 | INFO     | physioex.train.utils.fast_train:__init__:32 - Loading FastTrainDataset for parkinsons/night/PD with preprocess xsleepnet
2025-06-12 13:12:18.772 | INFO     | physioex.train.utils.fast_train:__init__:45 - Loaded X: torch.Size([63734, 3, 29, 129]), y: torch.Size([63734])
2025-06-12 13:12:30.210 | INFO     | physioex.train.utils.fast_train:__init__:70 - Loading FastEvalDataset for parkinsons/night/HOA with preprocess xsleepnet
2025-06-12 13:12:30.713 | INFO     | physioex.train.utils.fast_train:__init__:70 - Loading FastEvalDataset for parkinsons/night/PD with preprocess xsleepnet
2025-06-12 13:12:33.161 | INFO     | physioex.train.utils.fast_train:_